In [6]:
import urllib.error
import urllib.request
import pprint
from langchain.tools import tool

from langchain.chat_models import init_chat_model

import langchain_groq
import os

from dotenv import load_dotenv

load_dotenv()


True

In [14]:

model_groq_lamma70b = init_chat_model("llama-3.3-70b-versatile",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=10000, temperature=0.0)

# model = init_chat_model("openai/gpt-4o-mini",
#                         api_key=os.environ["OPENROUTER_API_KEY"],
#                         model_provider="openrouter",
#                         base_url="https://openrouter.ai/api/v1",
#                         max_tokens=1000, temperature=0.0)

# model = init_chat_model("nvidia/nemotron-3-ultra-550b-a55b:free",
#                         api_key=os.environ["OPENROUTER_API_KEY"],
#                         model_provider="openrouter",
#                         base_url="https://openrouter.ai/api/v1",
#                         max_tokens=1000, temperature=0.0)


# model = init_chat_model("openrouter/free",
#                         api_key=os.environ["OPENROUTER_API_KEY"],
#                         model_provider="openrouter",
#                         base_url="https://openrouter.ai/api/v1",
#                         max_tokens=1000, temperature=0.0)




In [8]:
model_or = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)


In [9]:
model_or_paid_gpt56_luna_pro = init_chat_model("openai/gpt-5.6-luna-pro",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)


In [10]:
response = model.invoke("which model are you?")

pprint.pprint("Model response:")
pprint.pprint(response)

'Model response:'
AIMessage(content='I am a Meta AI model, and my specific model name is Llama. Llama stands for "Large Language Model Meta AI."', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 40, 'total_tokens': 68, 'completion_time': 0.113340045, 'completion_tokens_details': None, 'prompt_time': 0.001913595, 'prompt_tokens_details': None, 'queue_time': 0.058499274, 'total_time': 0.11525364}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fdf63-728f-7f40-b426-d94102dd2cc5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 28, 'total_tokens': 68})


In [11]:
from langchain_core.tools import tool
import sqlite3

In [12]:

@tool
def save_trip_demo(user_id: str, destination: str) -> str:
    """Save a trip to the database. Irreversible without manual cleanup."""
    return f"Trip to {destination} saved for {user_id}."  # standing in for a real DB write

In [16]:
from langchain.agents import create_agent

from langchain.agents.middleware import SummarizationMiddleware


agent = create_agent(
    model=model_or_paid_gpt56_luna_pro,
    tools=[save_trip_demo],
    middleware=[
        SummarizationMiddleware(
            model = model_groq_lamma70b,
            trigger = ("tokens",4000)
        )
    ])

## HITL middleware

In [17]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def your_read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def your_send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model_groq_lamma70b,
    tools=[your_read_email_tool, your_send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "your_send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "your_read_email_tool": False,
            }
        ),
    ],
)

In [18]:
config = {'configurable':{"thread_id":"hitl"}}

result =agent.invoke({"messages": [("user", "Send an email to my manager on sunjtry@gmail.com, asking for a leave")]}, config=config)

In [19]:
result

{'messages': [HumanMessage(content='Send an email to my manager on sunjtry@gmail.com, asking for a leave', additional_kwargs={}, response_metadata={}, id='07902d44-9ed9-43c4-8047-d983b71aa4b5'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ydh6sg0rd', 'function': {'arguments': '{"body":"Dear Manager, I am writing to request a leave. Thank you.","recipient":"sunjtry@gmail.com","subject":"Leave Request"}', 'name': 'your_send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 312, 'total_tokens': 358, 'completion_time': 0.112307915, 'completion_tokens_details': None, 'prompt_time': 0.016317767, 'prompt_tokens_details': None, 'queue_time': 0.051488213, 'total_time': 0.128625682}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fdf72-292c-7613-b836-dd6ea50158

In [20]:
# --- Core LangChain ---
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.tools import tool as tool_rt, ToolRuntime

In [21]:
# --- LangGraph (checkpointing, resuming) ---
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

In [22]:
from langchain.agents.middleware import (
    SummarizationMiddleware,
    HumanInTheLoopMiddleware,
    ModelCallLimitMiddleware,
    ToolCallLimitMiddleware,
    ModelFallbackMiddleware,
    PIIMiddleware,
    TodoListMiddleware,
    LLMToolSelectorMiddleware,
    ToolRetryMiddleware,
    ModelRetryMiddleware,
    LLMToolEmulator,
    ContextEditingMiddleware,
    ClearToolUsesEdit,
)

Defining Tools for my Cinebot - Dummy as of now but ofcourse will be real as we have seen in last class.

In [23]:
@tool
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie at the cinema."""
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM only",
        "oppenheimer": "Sold out for tonight",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")

In [24]:

@tool
def book_seats(movie_title: str, seat_count: int) -> str:
    """Book seats for a movie. Irreversible once confirmed."""
    return f"Booked {seat_count} seat(s) for {movie_title}."

In [25]:
@tool
def cancel_booking(booking_id: str) -> str:
    """Cancel an existing booking. Irreversible."""
    return f"Booking {booking_id} cancelled."

In [26]:
@tool
def check_order_status(booking_id: str) -> str:
    """Check the status of an existing booking."""
    return f"Booking {booking_id}: confirmed, 2 seats, Interstellar, 7:00 PM."

In [27]:
@tool
def get_refund_policy() -> str:
    """Get the cinema's refund policy -- exact wording, not to be paraphrased."""
    return "Refunds available up to 2 hours before showtime. No refunds after that."

In [28]:

@tool
def lookup_seat_map(movie_title: str, seat_number: str) -> str:
    """Look up a specific seat -- fails if the seat number format is wrong."""
    if not seat_number or not seat_number[0].isalpha():
        raise ValueError(f"Malformed seat number '{seat_number}' -- expected a letter+number like 'A12'.")
    return f"Seat {seat_number} for {movie_title}: available."

In [29]:
cinebot_tools = [check_showtimes, book_seats, cancel_booking, check_order_status, get_refund_policy, lookup_seat_map]


### Summarization Middleware

In [35]:
summarizing_agent = create_agent(
    model=model_or,
    tools=cinebot_tools,
    middleware=[
        SummarizationMiddleware(
            model=model_or_paid_gpt56_luna_pro,
            trigger=('tokens', 300),
            keep=('messages', 3),
        )
    ]
)

In [36]:
result = summarizing_agent.invoke({"messages": [("user", "Is Interstellar showing tonight? also please make sure that you book me a ticket, refund me if it is not available,also share the refund policy for me to go through, also check my order status for book_1234")]})

In [37]:
result

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nFind out whether *Interstellar* is showing tonight, book the user a ticket if available, provide a refund if it is unavailable, share the refund policy, and check the order status for `book_1234`.\n\n## SUMMARY\n\n- Showtime lookup was performed for *Interstellar*.\n- Available showtimes returned: **7:00 PM** and **10:15 PM**.\n- No ticket has been booked yet.\n- No refund has been issued or evaluated because availability exists and no specific showtime, quantity, or booking details were provided.\n- The refund policy has not yet been retrieved or shared.\n- Order status for `book_1234` has not yet been checked.\n\n## ARTIFACTS\n\n- Showtime lookup tool call `check_showtimes_y9bvdyp42fd1` for movie title *Interstellar*.\n- Result: showtimes at **7:00 PM** and **10:15 PM**.\n- No files or other artifacts created or modified.\n\n## NEXT STEPS\n\n1. Ask the user which showtime they w

# HITL (Human in the Loop)

In [38]:
guarded_agent = create_agent(
    model=model_or,
    tools=cinebot_tools,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"cancel_booking": {"allowed_decisions": ["approve", "edit", "reject", "respond"]}}
        ),
    ],
    checkpointer=InMemorySaver(),  # REQUIRED -- HITL needs to pause and later resume
)

config = {'configurable':{'thread_id':'hitl-demo-live'}}

In [39]:
result = guarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)

In [40]:
result

{'messages': [HumanMessage(content='Please cancel booking BK1042', additional_kwargs={}, response_metadata={}, id='d9762a95-5c3a-4aa4-95f9-e5b962c20eb0'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call cancel_booking function.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'We need to call cancel_booking function.'}]}, response_metadata={'model_name': 'openai/gpt-oss-20b:free', 'id': 'gen-1786165279-qUgA4qHpnZ8rVZLxVg3V', 'created': 1786165279, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 2.45e-06, 'upstream_inference_prompt_cost': 4.0745e-06, 'upstream_inference_cost': 6.5245e-06}}, id='lc_run--019fdfbf-3971-7d40-a6a8-9008dd189b8c-0', tool_calls=[{'name': 'cancel_booking', 'args': {'booking_id': 'BK1042'}, 'id': 'call_577C7D30179D4BD68E8D88A5', 'type': 'tool_call'}], invalid_

In [42]:
from rich import print

In [43]:
print(result)

{
    'messages': [
        HumanMessage(
            content='Please cancel booking BK1042',
            additional_kwargs={},
            response_metadata={},
            id='d9762a95-5c3a-4aa4-95f9-e5b962c20eb0'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_content': 'We need to call cancel_booking function.',
                'reasoning_details': [
                    {
                        'type': 'reasoning.text',
                        'format': 'unknown',
                        'index': 0,
                        'text': 'We need to call cancel_booking function.'
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-oss-20b:free',
                'id': 'gen-1786165279-qUgA4qHpnZ8rVZLxVg3V',
                'created': 1786165279,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 2.45e-06,
                    'upstream_inference_prompt_cost': 4.0745e-06,
                    'upstream_inference_cost': 6.5245e-06
                }
            },
            id='lc_run--019fdfbf-3971-7d40-a6a8-9008dd189b8c-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'BK1042'},
                    'id': 'call_577C7D30179D4BD68E8D88A5',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 281,
                'output_tokens': 35,
                'total_tokens': 316,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 8}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'cancel_booking',
                        'args': {'booking_id': 'BK1042'},
                        'description': "Tool execution requires approval\n\nTool: cancel_booking\nArgs: 
{'booking_id': 'BK1042'}"
                    }
                ],
                'review_configs': [
                    {
                        'action_name': 'cancel_booking',
                        'allowed_decisions': ['approve', 'edit', 'reject', 'respond']
                    }
                ]
            },
            id='f08be289e18e4a510e40dc4179382ed0'
        )
    ]
}

## HITL Decision to resume agent

In [44]:
resumed_result = guarded_agent.invoke(Command(resume={"decisions":[{"type":"approve"}]}),config=config)

In [45]:
print(resumed_result)

{
    'messages': [
        HumanMessage(
            content='Please cancel booking BK1042',
            additional_kwargs={},
            response_metadata={},
            id='d9762a95-5c3a-4aa4-95f9-e5b962c20eb0'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_content': 'We need to call cancel_booking function.',
                'reasoning_details': [
                    {
                        'type': 'reasoning.text',
                        'format': 'unknown',
                        'index': 0,
                        'text': 'We need to call cancel_booking function.'
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-oss-20b:free',
                'id': 'gen-1786165279-qUgA4qHpnZ8rVZLxVg3V',
                'created': 1786165279,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 2.45e-06,
                    'upstream_inference_prompt_cost': 4.0745e-06,
                    'upstream_inference_cost': 6.5245e-06
                }
            },
            id='lc_run--019fdfbf-3971-7d40-a6a8-9008dd189b8c-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'BK1042'},
                    'id': 'call_577C7D30179D4BD68E8D88A5',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 281,
                'output_tokens': 35,
                'total_tokens': 316,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 8}
            }
        ),
        ToolMessage(
            content='Booking BK1042 cancelled.',
            name='cancel_booking',
            id='e324e2bc-fb4f-491c-9079-530fcc7895b8',
            tool_call_id='call_577C7D30179D4BD68E8D88A5'
        ),
        AIMessage(
            content='Your booking **BK1042** has been successfully cancelled.',
            additional_kwargs={
                'reasoning_content': 'The cancellation succeeded. We should inform the user.',
                'reasoning_details': [
                    {
                        'type': 'reasoning.text',
                        'format': 'unknown',
                        'index': 0,
                        'text': 'The cancellation succeeded. We should inform the user.'
                    }
                ]
            },
            response_metadata={
                'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free',
                'id': 'gen-1786165921-5j7AZZeI7SCXNRKYEO15',
                'created': 1786165921,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--019fdfc9-01f9-7450-885c-a32435fb64be-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 717,
                'output_tokens': 27,
                'total_tokens': 744,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 14}
            }
        )
    ]
}

In [46]:
def run_interactive_hitl_demo(agent, config):
    """A genuinely interactive HITL loop -- ask out loud, type the answer, watch it apply live."""
    state = agent.get_state(config)
    if not state.next:
        print("Nothing is currently paused for approval.")
        return

    print("The agent wants to call a guarded tool. Choose a decision:")
    print("  1) approve  -- run it exactly as proposed")
    print("  2) edit     -- run it, but change the booking_id first")
    print("  3) reject   -- block it, with a reason sent back to the agent")
    print("  4) respond  -- answer a question instead of deciding on the action")

    choice = input("Type 1, 2, 3, or 4: ").strip()

    if choice == "1":
        decision = {"type": "approve"}
    elif choice == "2":
        new_id = input("New booking_id to use instead: ").strip()
        decision = {"type": "edit", "args": {"booking_id": new_id}}
    elif choice == "3":
        reason = input("Reason for rejecting: ").strip()
        decision = {"type": "reject", "message": reason}
    elif choice == "4":
        answer = input("Your response to the agent: ").strip()
        decision = {"type": "respond", "message": answer}
    else:
        print("Not a valid choice -- try again.")
        return

    resumed = agent.invoke(Command(resume={"decisions": [decision]}), config=config)
    print()
    print("Agent's final response:", resumed["messages"][-1].content)


In [47]:
config = {'configurable':{'thread_id':'hitl-demo-live-2'}}

In [ ]:
newguarded_agent = create_agent(
    model=model_or,
    tools=cinebot_tools,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"cancel_booking": {"allowed_decisions": ["approve", "edit", "reject", "respond"]}}
        ),
    ],
    checkpointer=InMemorySaver(),  # REQUIRED -- HITL needs to pause and later resume
)

In [56]:
newresult = newguarded_agent.invoke({"messages": [("user", "Please cancel booking BK1042")]}, config=config)

In [57]:
run_interactive_hitl_demo(newguarded_agent, config)

The agent wants to call a guarded tool. Choose a decision:

1) approve  -- run it exactly as proposed

2) edit     -- run it, but change the booking_id first

3) reject   -- block it, with a reason sent back to the agent

4) respond  -- answer a question instead of deciding on the action

Agent's final response: I couldn’t cancel booking **BK1042**. It is still **confirmed** for **2 seats** to 
*Interstellar* at **7:00 PM**.

⚙️ Code Walkthrough: Command(resume={"decisions": [...]}) is how you hand a decision back to an agent that's paused mid-run. The decisions list has one entry per interrupted tool call (usually just one). Each decision is a dict with a "type" key matching one of the four options, plus whatever extra data that type needs — edit needs new args, reject and respond need a message, approve needs nothing else.

# Model Call Limit

In [58]:
call_limited_agent = create_agent(
    model=model_or,
    tools=cinebot_tools,
    checkpointer=InMemorySaver(),  # required for thread_limit to persist across calls
    middleware=[
        ModelCallLimitMiddleware(
            thread_limit=5,   # across the WHOLE conversation
            run_limit=1,       # per single .invoke() call
            exit_behavior="end",  # graceful stop, not an exception
        ),
    ],
)

In [59]:
result = call_limited_agent.invoke(
    {"messages": [("user", "What's showing tonight?")]},
    config={"configurable": {"thread_id": "call-limit-demo"}},
)

In [60]:
print(result)

{
    'messages': [
        HumanMessage(
            content="What's showing tonight?",
            additional_kwargs={},
            response_metadata={},
            id='48ac015e-92e6-49db-9416-726f1cea35ab'
        ),
        AIMessage(
            content="I'd be happy to check what's showing tonight! However, I need to know which specific movie 
you're interested in to look up the showtimes. \n\nCould you let me know the name of the movie you'd like to see? 
Or if you're not sure what's playing, you might want to check the cinema's website or app for a full listing of 
tonight's showtimes.",
            additional_kwargs={},
            response_metadata={
                'model_name': 'poolside/laguna-xs-2.1:free',
                'id': 'gen-1786167936-JkDF6lLtz0WqkM0b7SNG',
                'created': 1786167936,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--019fdfe7-c231-7d23-a9ef-c1cfee66e226-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 472,
                'output_tokens': 81,
                'total_tokens': 553,
                'input_token_details': {'cache_read': 16, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        )
    ]
}

In [61]:
result_invoke_2= call_limited_agent.invoke(
    {"messages": [("user", "cancel my booking B123? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)
print(result_invoke_2)

{
    'messages': [
        HumanMessage(
            content='cancel my booking B123? ',
            additional_kwargs={},
            response_metadata={},
            id='e92f2dfb-c7d4-4292-8363-88ecd1ff9208'
        ),
        AIMessage(
            content="I'll cancel your booking B123 for you.",
            additional_kwargs={},
            response_metadata={
                'model_name': 'poolside/laguna-xs-2.1:free',
                'id': 'gen-1786168351-1GT40QPuqQCoX01ydPnn',
                'created': 1786168351,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--019fdfee-18d9-73d1-b1a8-07f6c92cb95a-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B123'},
                    'id': 'chatcmpl-tool-bdbb506da49c7768',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 477,
                'output_tokens': 38,
                'total_tokens': 515,
                'input_token_details': {'cache_read': 16, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Booking B123 cancelled.',
            name='cancel_booking',
            id='fdfe47ea-e0f7-4a90-a55b-7e2e7aed5b78',
            tool_call_id='chatcmpl-tool-bdbb506da49c7768'
        ),
        AIMessage(
            content='Model call limits exceeded: run limit (1/1)',
            additional_kwargs={},
            response_metadata={},
            id='af8f1d44-fe9b-4913-9fd8-ba34166efbdc',
            tool_calls=[],
            invalid_tool_calls=[]
        )
    ]
}

In [62]:
result_invoke3 = call_limited_agent.invoke(
    {"messages": [("user", "What all movies are being shown? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)
print(result_invoke3)

{
    'messages': [
        HumanMessage(
            content='cancel my booking B123? ',
            additional_kwargs={},
            response_metadata={},
            id='e92f2dfb-c7d4-4292-8363-88ecd1ff9208'
        ),
        AIMessage(
            content="I'll cancel your booking B123 for you.",
            additional_kwargs={},
            response_metadata={
                'model_name': 'poolside/laguna-xs-2.1:free',
                'id': 'gen-1786168351-1GT40QPuqQCoX01ydPnn',
                'created': 1786168351,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--019fdfee-18d9-73d1-b1a8-07f6c92cb95a-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B123'},
                    'id': 'chatcmpl-tool-bdbb506da49c7768',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 477,
                'output_tokens': 38,
                'total_tokens': 515,
                'input_token_details': {'cache_creation': 0, 'cache_read': 16},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Booking B123 cancelled.',
            name='cancel_booking',
            id='fdfe47ea-e0f7-4a90-a55b-7e2e7aed5b78',
            tool_call_id='chatcmpl-tool-bdbb506da49c7768'
        ),
        AIMessage(
            content='Model call limits exceeded: run limit (1/1)',
            additional_kwargs={},
            response_metadata={},
            id='af8f1d44-fe9b-4913-9fd8-ba34166efbdc',
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(
            content='What all movies are being shown? ',
            additional_kwargs={},
            response_metadata={},
            id='73e19192-ca69-48c3-88c1-5595edf3a4ed'
        ),
        AIMessage(
            content="I don't have a function to list all movies currently being shown at the cinema. My available 
tools only allow me to check showtimes for a specific movie when you provide its title.\n\nTo find out what movies 
are playing, you might need to:\n- Check the cinema's website or app directly\n- Visit the cinema's lobby or box 
office\n- Ask me about a specific movie you're interested in, and I can check its showtimes\n\nIs there a 
particular movie you'd like to know the showtimes for?",
            additional_kwargs={},
            response_metadata={
                'model_name': 'poolside/laguna-xs-2.1:free',
                'id': 'gen-1786168375-mBJxHtB7cE4qGCmyM8ZH',
                'created': 1786168375,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--019fdfee-7694-7940-a42d-e0372660e397-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 564,
                'output_tokens': 108,
                'total_tokens': 672,
                'input_token_details': {'cache_read': 16, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        )
    ]
}

In [63]:
result_invoke4 = call_limited_agent.invoke(
    {"messages": [("user", "Summarize my chat? ")]},
    config={"configurable": {"thread_id": "call-limit-demo-4"}},
)

In [64]:
print(result_invoke4)

{
    'messages': [
        HumanMessage(
            content='cancel my booking B123? ',
            additional_kwargs={},
            response_metadata={},
            id='e92f2dfb-c7d4-4292-8363-88ecd1ff9208'
        ),
        AIMessage(
            content="I'll cancel your booking B123 for you.",
            additional_kwargs={},
            response_metadata={
                'model_name': 'poolside/laguna-xs-2.1:free',
                'id': 'gen-1786168351-1GT40QPuqQCoX01ydPnn',
                'created': 1786168351,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--019fdfee-18d9-73d1-b1a8-07f6c92cb95a-0',
            tool_calls=[
                {
                    'name': 'cancel_booking',
                    'args': {'booking_id': 'B123'},
                    'id': 'chatcmpl-tool-bdbb506da49c7768',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 477,
                'output_tokens': 38,
                'total_tokens': 515,
                'input_token_details': {'cache_creation': 0, 'cache_read': 16},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='Booking B123 cancelled.',
            name='cancel_booking',
            id='fdfe47ea-e0f7-4a90-a55b-7e2e7aed5b78',
            tool_call_id='chatcmpl-tool-bdbb506da49c7768'
        ),
        AIMessage(
            content='Model call limits exceeded: run limit (1/1)',
            additional_kwargs={},
            response_metadata={},
            id='af8f1d44-fe9b-4913-9fd8-ba34166efbdc',
            tool_calls=[],
            invalid_tool_calls=[]
        ),
        HumanMessage(
            content='What all movies are being shown? ',
            additional_kwargs={},
            response_metadata={},
            id='73e19192-ca69-48c3-88c1-5595edf3a4ed'
        ),
        AIMessage(
            content="I don't have a function to list all movies currently being shown at the cinema. My available 
tools only allow me to check showtimes for a specific movie when you provide its title.\n\nTo find out what movies 
are playing, you might need to:\n- Check the cinema's website or app directly\n- Visit the cinema's lobby or box 
office\n- Ask me about a specific movie you're interested in, and I can check its showtimes\n\nIs there a 
particular movie you'd like to know the showtimes for?",
            additional_kwargs={},
            response_metadata={
                'model_name': 'poolside/laguna-xs-2.1:free',
                'id': 'gen-1786168375-mBJxHtB7cE4qGCmyM8ZH',
                'created': 1786168375,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--019fdfee-7694-7940-a42d-e0372660e397-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 564,
                'output_tokens': 108,
                'total_tokens': 672,
                'input_token_details': {'cache_creation': 0, 'cache_read': 16},
                'output_token_details': {'reasoning': 0}
            }
        ),
      

# Model Fallback

In [65]:
resilient_agent = create_agent(
    model="openai:gpt-5.5-haiku",     # primary, most capable
    tools=cinebot_tools,
)

In [66]:
result = resilient_agent.invoke( {"messages": [("user", "Summarize my chat? ")]},)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `gpt-5.5-haiku` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

In [67]:
resilient_agent = create_agent(
    model="openai:gpt-5.5-haiku",     # primary, most capable assume its not available or decomissioned, so we fall back to a cheaper model
    tools=cinebot_tools,
    middleware=[
        ModelFallbackMiddleware(
            model_or,   # fallback -- cheaper, still OpenAI, needs no extra setup
            model_groq_lamma70b,   # a further, fully-local last resort -- uncomment if you have
                                    # `pip install langchain-ollama` AND a local Ollama server running.
                                    # Left commented here so this cell runs with nothing beyond
                                    # what Setup already installed.
        ),
    ],
)
print("Fallback chain: gpt-5.5 haiku -> gpt-5-mini.")
print("If the primary model call fails for any reason, this silently tries the next one.")

Fallback chain: gpt-5.5 haiku -> gpt-5-mini.

If the primary model call fails for any reason, this silently tries the next one.

In [68]:
result = resilient_agent.invoke( {"messages": [("user", "Summarize my chat? ")]},)

In [69]:
print(result)

{
    'messages': [
        HumanMessage(
            content='Summarize my chat? ',
            additional_kwargs={},
            response_metadata={},
            id='ba773fa4-4d03-4f36-a7a9-ebaadb2f2878'
        ),
        AIMessage(
            content="I don't have any previous conversation history to summarize. This is the start of our chat, so
there's nothing to summarize yet!\n\nIf you'd like me to summarize something, please share the chat content you'd 
like me to summarize.",
            additional_kwargs={
                'reasoning_content': 'The user is asking me to summarize their chat. However, looking at the 
conversation, the user has only sent one message: "Summarize my chat?" - there\'s no previous chat history to 
summarize. I should let them know that I don\'t have any previous conversation history to summarize.',
                'reasoning_details': [
                    {
                        'type': 'reasoning.text',
                        'format': 'unknown',
                        'index': 0,
                        'text': 'The user is asking me to summarize their chat. However, looking at the 
conversation, the user has only sent one message: "Summarize my chat?" - there\'s no previous chat history to 
summarize. I should let them know that I don\'t have any previous conversation history to summarize.'
                    }
                ]
            },
            response_metadata={
                'model_name': 'inclusionai/ling-3.0-tiny:free',
                'id': 'gen-1786168580-jKmjybHnOodcxapRUc9a',
                'created': 1786168580,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--019fdff1-951d-77c0-8a5f-6d29f68607f9-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 648,
                'output_tokens': 110,
                'total_tokens': 758,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 70}
            }
        )
    ]
}

# Tool Call Limit

In [70]:
# Tool Call Limit

In [71]:
tool_limited_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=cinebot_tools,
    checkpointer=InMemorySaver(),
    middleware=[
        ToolCallLimitMiddleware(run_limit=8),                              # global, this turn
        ToolCallLimitMiddleware(tool_name="cancel_booking", thread_limit=2),  # tighter, one tool, whole conversation
    ],
)